# sam4xtal setup (JupyterHub)

Run **every cell top to bottom**. You do not need a terminal.

What this does:

1. Saves your Hugging Face token
2. Creates a local runtime folder next to this notebook (`sam4xtal-runtime/`) for the model cache + Python venv
3. Installs the SAM3 sidecar (GPU inference API)
4. Builds and starts the **Next.js** website
5. Prints the **URL / port** you should open in your browser

No Docker. First run can take a long time (downloads).

**Before you start:** open a JupyterHub session with a **GPU** if you want real SAM3 (not the mock).  
Clone/copy this repo under `$SCRATCH` or `$WORK` if your home quota is small — but the **model cache always goes into `notebooks/sam4xtal-runtime/`** next to this file.

## 1. Your Hugging Face token

1. Open https://huggingface.co/settings/tokens and create a token (read is enough)
2. Accept the license for https://huggingface.co/facebook/sam3
3. Paste the token below (replace the placeholder), then run the cell

In [ ]:
# <<< PASTE YOUR TOKEN BETWEEN THE QUOTES >>>
HF_TOKEN = "PASTE_YOUR_TOKEN_HERE"

# Ports (change only if something else already uses them)
SIDECAR_PORT = 9001   # SAM3 API (internal)
WEB_PORT = 3000       # Next.js UI — this is what you open

# True = no GPU / no model weights (smoke test only)
USE_MOCK = False

## 2. Locate the project + create runtime folder

Everything heavy (HF models, venv, logs) goes into `sam4xtal-runtime/` next to this notebook.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = None
for candidate in [HERE, HERE.parent, *HERE.parents]:
    if (candidate / "jupyterhub" / "sam4xtal_hub" / "setup_runtime.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the sam4xtal repo. Open this notebook from the cloned project "
        "(…/sam4xtal/notebooks/setup.ipynb)."
    )

hub = str(ROOT / "jupyterhub")
if hub not in sys.path:
    sys.path.insert(0, hub)

from sam4xtal_hub.setup_runtime import HubRuntime

# Prefer the folder that actually contains setup.ipynb
nb_dir = HERE if (HERE / "setup.ipynb").is_file() else (ROOT / "notebooks")

rt = HubRuntime(
    notebook_directory=nb_dir,
    sidecar_port=SIDECAR_PORT,
    web_port=WEB_PORT,
    mock=USE_MOCK,
)
rt.prepare_dirs()
if not USE_MOCK:
    rt.save_hf_token(HF_TOKEN)
else:
    print("USE_MOCK=True — skipping HF token")
rt.write_hub_env()
print("OK")

## 3. Install Python sidecar (uv + venv + torch)

First time: several GB download. Stay on a GPU node for the real model.

In [ ]:
rt.setup_sidecar_venv()
print("sidecar environment ready")

## 4. Start the SAM3 sidecar

Loads / downloads weights into `sam4xtal-runtime/hf-cache/`. Keep this cell running until it prints **ready**.

In [ ]:
client = rt.start_sidecar(restart=True)
print(client.health())

## 5. Install + build the Next.js frontend

Installs **Vite+ (`vp`)** into `sam4xtal-runtime/vite-plus/` (managed Node + pnpm).  
Does **not** use the read-only module Node under `/software/opt/…` (corepack/`npm -g` fail there).

In [ ]:
rt.setup_web()
print("frontend build ready")

## 6. Start the website

Starts `next start` on **port 3000** (or your `WEB_PORT`).

In [ ]:
rt.start_web(restart=True)
print(rt.status())

## 7. Open this URL

Run the cell and **read the printed instructions**.  
The UI is the Next.js app (same as on a normal machine). The sidecar stays on localhost inside this session.

In [ ]:
rt.print_access_instructions()

info = rt.access_urls()
print()
print("Quick copy:")
print(f"  web port     = {info['web_port']}")
print(f"  local URL    = {info['local_web']}")
print(f"  this node    = {info['node_web']}")
if info["proxy_paths"]:
    print(f"  hub proxy    = {info['proxy_paths'][0]}")

### If the page does not load in the Hub tab

JupyterHub often **does not** expose port 3000 automatically. Then use an SSH tunnel from your laptop (with VPN if off-campus):

```bash
ssh -L 3000:127.0.0.1:3000 YOUR_USERNAME@HOSTNAME_FROM_CELL_ABOVE
```

Open http://127.0.0.1:3000 in your laptop browser.  
Hostname is printed above (and matches `socket.gethostname()` on the Hub node).

Usage after that is the same as the normal app: load SEM images, click crystals, segment, save annotations.

## 8. Status / logs (optional)

In [ ]:
import json
print(json.dumps(rt.status(), indent=2, default=str))
print("\n--- sidecar log (tail) ---")
rt.tail_log(rt.sidecar_log, 20)
print("\n--- web log (tail) ---")
rt.tail_log(rt.web_log, 20)

## 9. Stop everything (when you are done)

Also **stop / shut down the JupyterHub server** so the GPU is released.

In [ ]:
rt.stop_all()